In [1]:
# Import required modules
!pip install -U langchain langchain-community pypdf
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# We'll use a simple HuggingFace LLM (free, no API key needed)
from langchain_community.llms import HuggingFacePipeline

from transformers import pipeline

In [2]:
# Upload your PDF file manually

from google.colab import files
uploaded = files.upload()

# Get the uploaded file name
pdf_file = list(uploaded.keys())[0]
print("Uploaded file:", pdf_file)

Saving intro-to-ml.pdf to intro-to-ml (1).pdf
Uploaded file: intro-to-ml (1).pdf


In [3]:
# Load the PDF using LangChain loader

loader = PyPDFLoader(pdf_file)
documents = loader.load()

# Print number of pages
print("Total pages:", len(documents))

# Show sample text
print("\nSample page content:\n")
print(documents[0].page_content[:500])

Total pages: 392

Sample page content:

Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning  
with P y t h o n   
A GUIDE FOR DATA SCIENTISTS


In [4]:
# Split large text into smaller chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     # max size of each chunk
    chunk_overlap=100   # overlap for better context
)

chunks = text_splitter.split_documents(documents)

print("Total chunks created:", len(chunks))

Total chunks created: 1137


In [5]:
# Load embedding model

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/tmp/ipykernel_12004/2175454312.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
!pip install faiss-cpu

In [7]:
# Store chunks in FAISS vector database

vector_db = FAISS.from_documents(chunks, embedding_model)

print("Vector DB created successfully!")

Vector DB created successfully!


In [8]:
# Test similarity search

query = "What is supervised learning?"

results = vector_db.similarity_search(query, k=3)

# Print retrieved chunks
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---\n")
    print(doc.page_content[:300])


--- Result 1 ---

CHAPTER 2
Supervised Learning
As we mentioned earlier, supervised machine learning is one of the most commonly
used and successful types of machine learning. In this chapter, we will describe super‐
vised learning in more detail and explain several popular supervised learning algo‐
rithms. We alread

--- Result 2 ---

2. Supervised Learning. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .  25
Classification and Regression                                                                                         25
Generalization, Overfitting, and Underfit

--- Result 3 ---

chapter, we will go into more depth about the different kinds of supervised models in
scikit-learn and how to apply them successfully.
24 | Chapter 1: Introduction


In [9]:
!pip install -U transformers

In [16]:
model_id="tiiuae/falcon-7b"

In [18]:
# Create a text generation pipeline

from transformers import pipeline
import torch

qa_pipeline = pipeline("text-generation",
                                  model=model_id,
                                  model_kwargs={'dtype':torch.bfloat16},
                                  max_new_tokens=200,
                                  device=0,
                                  temperature=0.7)

# Wrap it with LangChain
llm = HuggingFacePipeline(pipeline=qa_pipeline)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [19]:
# RAG pipeline function

def rag_pipeline(query, k=5):

    # Step 1: Retrieve relevant chunks
    docs = vector_db.similarity_search(query, k=k)

    # Step 2: Combine retrieved text
    context = "\n\n".join([d.page_content for d in docs])

    # Step 3: Improved prompt (VERY IMPORTANT)
    prompt = f"""
    You are a helpful assistant.

    Answer the question in ONLY 2-3 concise sentences.Don't try to make your own answers.if you don't know the answer,just say you don't know.
    Do NOT repeat the question.
    Do NOT generate extra text.
    Use ONLY the given context for generating answers.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    # Step 4: Generate response
    response = llm.invoke(prompt)

    # Step 5: Clean output (IMPORTANT FIX)
    if isinstance(response, list):
        answer = response[0]['generated_text']
    else:
        answer = str(response)

    # Remove prompt from output (common issue)
    answer = answer.replace(prompt, "").strip()

    return answer

In [ ]:
# Test queries

queries = [
    "What is supervised learning?",
    "Explain k-nearest neighbors",
    "What is overfitting?",
    "What is unsupervised learning?"
]

for q in queries:
    print("\n============================")
    print("Question:", q)
    print("Answer:", rag_pipeline(q))


Question: What is supervised learning?


[transformers] Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
# Real-time chatbot loop

while True:
    query = input("\nAsk a question (type 'exit' to stop): ")

    if query.lower() == "exit":
        break

    answer = rag_pipeline(query)
    print("\nAnswer:", answer)